# `MC_cable`: sweep debug

This notebook loads a finished sweep artifact directory, inspects `case_metrics.csv`, and renders compact summary plots with an emphasis on error distributions.


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

os.environ.setdefault("JAX_PLATFORMS", "cpu")


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
artifact_dir = repo_root / "examples" / "neuron_compare" / "MC_cable" / "artifacts" / "cable_voltage_smoke"
metrics_path = artifact_dir / "case_metrics.csv"
aggregate_path = artifact_dir / "aggregate.json"

print("artifact_dir:", artifact_dir)
print("metrics_path exists:", metrics_path.exists())
print("aggregate_path exists:", aggregate_path.exists())


In [ ]:
df = pd.read_csv(metrics_path)
ok_df = df[df["status"] == "ok"].copy()
ok_df["swc_name"] = ok_df["swc_path"].map(lambda value: Path(value).name)
ok_df["swc_cv_label"] = ok_df.apply(lambda row: f"{row['swc_name']}\ncv={int(row['cv_per_branch'])}", axis=1)
aggregate = json.loads(aggregate_path.read_text())

print(json.dumps(aggregate, indent=2, sort_keys=True))
ok_df.head()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10), constrained_layout=True)

mae_groups = [group["mae"].values for _, group in ok_df.groupby("swc_name", sort=True)]
mae_labels = list(ok_df.groupby("swc_name", sort=True).groups.keys())
box1 = axes[0].boxplot(mae_groups, patch_artist=True, labels=mae_labels, showmeans=True)
for patch in box1["boxes"]:
    patch.set_facecolor("#cfe8ff")
    patch.set_edgecolor("#264653")
for median in box1["medians"]:
    median.set_color("#d1495b")
    median.set_linewidth(1.5)
axes[0].set_title("MAE distribution by SWC")
axes[0].set_ylabel("MAE (mV)")
axes[0].grid(axis="y", alpha=0.25)

max_groups = [group["max_abs"].values for _, group in ok_df.groupby("swc_cv_label", sort=True)]
max_labels = list(ok_df.groupby("swc_cv_label", sort=True).groups.keys())
box2 = axes[1].boxplot(max_groups, patch_artist=True, labels=max_labels, showmeans=True)
for patch in box2["boxes"]:
    patch.set_facecolor("#fde0c5")
    patch.set_edgecolor("#264653")
for median in box2["medians"]:
    median.set_color("#d1495b")
    median.set_linewidth(1.5)
axes[1].set_title("Max absolute error by SWC and CV policy")
axes[1].set_ylabel("Max |delta V| (mV)")
axes[1].grid(axis="y", alpha=0.25)
axes[1].tick_params(axis="x", rotation=15)

plt.show()


In [ ]:
ranked = ok_df.sort_values(["mae", "max_abs"], ascending=False)
ranked[["case_id", "swc_name", "cv_per_branch", "dt_ms", "stimulus_kind", "mae", "rmse", "max_abs", "rel_mae_pct"]]
